# Science Nature Wiki v1 Builder

Build `science_nature_wiki_v1`, the encyclopedic/definition-oriented Wikipedia corpus for the Science/Nature RAG.

Design:

- `SciQ facts` handles short school-style facts and practical reasoning.
- `science_nature_wiki_v1` handles definitions, concepts, processes, theories, and broad scientific entities.
- Discovery is controlled with Wikidata macro-areas and quotas.
- Article text is fetched from `DragonLLM/Clean-Wikipedia-English-Articles` by Wikidata QID.
- Index chunks use 512 tokens with 128 overlap.

In [ ]:
# @title Mount Google Drive

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/NLP")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Non trovo la cartella progetto. "
        "Controlla se il path è /content/drive/MyDrive/NLP oppure modifica PROJECT_ROOT manualmente."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

## 0. Setup

If running in Colab and dependencies are missing, uncomment the install cell.

In [ ]:
# !pip -q install datasets pandas tqdm langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu tiktoken

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil
import time
import urllib.parse
import urllib.request

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from tqdm.auto import tqdm

# Example for Colab/Drive:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/NLP"
PROJECT_ROOT_OVERRIDE = None

PROJECT_MARKERS = [Path("requirements.txt"), Path("millionaire_client")]
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path("/content/drive/MyDrive/NLP"),
    Path("/content/drive/MyDrive/Colab Notebooks/NLP"),
    Path("/gdrive/MyDrive/NLP"),
    Path("/gdrive/MyDrive/Colab Notebooks/NLP"),
]

def looks_like_project_root(path):
    return any((path / marker).exists() for marker in PROJECT_MARKERS)

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((candidate for candidate in PROJECT_ROOT_CANDIDATES if looks_like_project_root(candidate)), Path.cwd())

DATASETS_DIR = PROJECT_ROOT / "Datasets"
INDEXES_DIR = PROJECT_ROOT / "Indexes"
LOGS_DIR = PROJECT_ROOT / "logs"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
INDEXES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

RUN_WIKIDATA_DISCOVERY = False
RUN_FETCH_AND_BUILD_DATASET = False  # inspect selected QIDs first, then turn True
RUN_INDEX_BUILD = True

SOURCE_WIKI_DATASET = "DragonLLM/Clean-Wikipedia-English-Articles"
OUTPUT_DATASET_DIR = DATASETS_DIR / "science_nature_wiki_v1"
OUTPUT_INDEX_DIR = INDEXES_DIR / "science_nature_wiki_v1"

CANDIDATE_QIDS_CSV = LOGS_DIR / "science_nature_wiki_v1_candidate_qids.csv"
SELECTED_QIDS_CSV = LOGS_DIR / "science_nature_wiki_v1_selected_qids.csv"
FETCH_REPORT_CSV = LOGS_DIR / "science_nature_wiki_v1_fetch_report.csv"
BUILD_REPORT_JSON = LOGS_DIR / "science_nature_wiki_v1_build_report.json"
INDEX_REPORT_JSON = LOGS_DIR / "science_nature_wiki_v1_index_report.json"
WIKIDATA_CACHE_JSON = LOGS_DIR / "science_nature_wiki_v1_wikidata_cache.json"

USER_AGENT = "science-nature-wiki-v1-builder/1.0 (student NLP project; Wikidata + DragonLLM)"
WIKIPEDIA_LICENSE = "cc-by-sa-4.0"
WIKI_SOURCE_TYPE = "wikipedia_article"
TAXONOMY_SOURCE = "wikidata_science_qid_v1"

OVERWRITE_OUTPUT_DATASET = False
OVERWRITE_INDEX = False
MAX_SOURCE_ROWS_TO_SCAN = None  # None = scan DragonLLM stream until all targets found or source ends

print("project root:", PROJECT_ROOT)
print("output dataset:", OUTPUT_DATASET_DIR)
print("output index:", OUTPUT_INDEX_DIR)


## 1. Taxonomy, Quotas, Manual Seeds

The same macro-labels used by SciQ are kept here. `topic` is the Wikipedia title / Wikidata label.

In [ ]:
SUBJECT_QUOTAS = {
    "physics": 400,
    "chemistry": 400,
    "biology": 550,
    "ecology": 300,
    "earth_science": 150,
    "astronomy": 150,
    "scientific_method": 80,
    "general_science": 100,
}

# Root QIDs are intentionally broad. Quotas and filters keep v1 controlled.
SUBJECT_ROOT_QIDS = {
    "physics": ["Q413"],
    "chemistry": ["Q2329"],
    "biology": ["Q420"],
    "ecology": ["Q7150"],
    "earth_science": ["Q8008", "Q1069", "Q25261", "Q521", "Q161574"],
    "astronomy": ["Q333"],
    "scientific_method": ["Q1438073", "Q101965", "Q12453", "Q41719"],
    "general_science": ["Q336", "Q7991"],
}

MANUAL_SEED_TITLES = {
    "physics": [
        "Physics", "Force", "Motion", "Friction", "Gravity", "Mass", "Weight", "Energy", "Kinetic energy",
        "Potential energy", "Momentum", "Inertia", "Acceleration", "Velocity", "Speed", "Pressure", "Temperature",
        "Heat", "Light", "Sound", "Wave", "Electric charge", "Electric current", "Electricity", "Magnetism",
        "Electromagnetism", "Simple machine", "String theory", "Quantum mechanics", "Thermodynamics",
    ],
    "chemistry": [
        "Chemistry", "Atom", "Electron", "Proton", "Neutron", "Molecule", "Chemical element", "Compound (chemistry)",
        "Chemical bond", "Ionic bonding", "Covalent bond", "Ion", "Sodium", "Chlorine", "Sodium chloride",
        "Acid", "Base (chemistry)", "pH", "Chemical reaction", "Reactant", "Product (chemistry)", "Solution (chemistry)",
        "Periodic table", "Metal", "Nonmetal", "Mixture", "States of matter",
    ],
    "biology": [
        "Biology", "Life", "Cell (biology)", "Cell nucleus", "Cell membrane", "Mitochondrion", "DNA", "Gene",
        "Protein", "Enzyme", "Photosynthesis", "Cellular respiration", "Evolution", "Natural selection",
        "Organism", "Tissue (biology)", "Organ (biology)", "Plant", "Animal", "Bacteria", "Virus", "Fungus",
        "Mammal", "Bird", "Reptile", "Amphibian", "Fish", "Flowering plant", "Seed", "Root", "Leaf",
    ],
    "ecology": [
        "Ecology", "Ecosystem", "Habitat", "Population ecology", "Community (ecology)", "Food chain", "Food web",
        "Predation", "Predator", "Prey", "Producer (biology)", "Consumer (food chain)", "Decomposer", "Biodiversity",
        "Biome", "Competition (biology)", "Ecological niche", "Adaptation", "Conservation biology", "Pollution",
        "Forest", "Desert", "Aquatic ecosystem", "Woodpecker",
    ],
    "earth_science": [
        "Earth science", "Earth", "Geology", "Rock (geology)", "Mineral", "Soil", "Erosion", "Weathering",
        "Volcano", "Earthquake", "Plate tectonics", "Fossil", "Sedimentary rock", "Igneous rock", "Metamorphic rock",
        "Atmosphere of Earth", "Weather", "Climate", "Water cycle", "Ocean", "River", "Glacier", "Mountain",
    ],
    "astronomy": [
        "Astronomy", "Universe", "Galaxy", "Star", "Sun", "Moon", "Planet", "Solar System", "Orbit", "Comet",
        "Asteroid", "Meteorite", "Telescope", "Constellation", "Light-year", "Gravity", "Black hole", "Nebula",
    ],
    "scientific_method": [
        "Scientific method", "Hypothesis", "Experiment", "Observation", "Measurement", "Data", "Evidence",
        "Theory", "Scientific theory", "Variable (research)", "Control group", "Model organism", "Peer review",
    ],
    "general_science": [
        "Science", "Natural science", "Nature", "Matter", "System", "Cause and effect", "Classification", "Measurement",
        "Scientific law", "Scientific modelling", "Research", "Laboratory", "Technology",
    ],
}

print("target article total:", sum(SUBJECT_QUOTAS.values()))
display(pd.DataFrame([{"subject": k, "quota": v, "roots": ", ".join(SUBJECT_ROOT_QIDS[k])} for k, v in SUBJECT_QUOTAS.items()]))


## 2. Discovery Helpers

Manual seeds are resolved through Wikipedia pageprops. Broad candidate discovery uses Wikidata SPARQL and enwiki sitelinks.

In [ ]:
QID_RE = re.compile(r"Q\d+")
SPACE_RE = re.compile(r"\s+")

def normalize_text(value):
    if value is None:
        return ""
    return SPACE_RE.sub(" ", str(value).replace("\u00a0", " ")).strip()

def clean_qid(value):
    if isinstance(value, dict):
        for key in ["id", "qid", "value", "entity"]:
            if key in value:
                found = clean_qid(value[key])
                if found:
                    return found
    if isinstance(value, (list, tuple)):
        for item in value:
            found = clean_qid(item)
            if found:
                return found
    match = QID_RE.search(str(value or ""))
    return match.group(0) if match else ""

def normalize_title(value):
    title = normalize_text(value).replace("_", " ")
    return title.lower()

def canonical_url(value):
    text = normalize_text(value)
    if not text:
        return ""
    return text.split("#", 1)[0]

def safe_int(value, default=0):
    try:
        return int(float(value))
    except Exception:
        return default

def load_cache(path):
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def save_cache(path, cache):
    path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")

wikidata_cache = load_cache(WIKIDATA_CACHE_JSON)

def http_json(url, timeout=60, max_retries=6, base_sleep=2.0):
    req = urllib.request.Request(
        url,
        headers={"User-Agent": USER_AGENT, "Accept": "application/json"},
    )
    last_exc = None

    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as response:
                return json.loads(response.read().decode("utf-8"))

        except urllib.error.HTTPError as exc:
            last_exc = exc
            if exc.code not in (429, 500, 502, 503, 504):
                raise

            retry_after = exc.headers.get("Retry-After")
            if retry_after:
                try:
                    sleep_seconds = float(retry_after)
                except Exception:
                    sleep_seconds = base_sleep * (2 ** attempt)
            else:
                sleep_seconds = base_sleep * (2 ** attempt)

            print(
                f"HTTP {exc.code}; sleeping {sleep_seconds:.1f}s "
                f"then retrying ({attempt + 1}/{max_retries})"
            )
            time.sleep(sleep_seconds)

        except urllib.error.URLError as exc:
            last_exc = exc
            sleep_seconds = base_sleep * (2 ** attempt)
            print(
                f"URL error {exc}; sleeping {sleep_seconds:.1f}s "
                f"then retrying ({attempt + 1}/{max_retries})"
            )
            time.sleep(sleep_seconds)

    raise last_exc

def run_sparql(query_name, query, sleep_seconds=0.25):
    if query_name in wikidata_cache:
        return wikidata_cache[query_name]
    params = urllib.parse.urlencode({"query": query, "format": "json"})
    url = "https://query.wikidata.org/sparql?" + params
    data = http_json(url, timeout=120)
    rows = []
    for binding in data.get("results", {}).get("bindings", []):
        item = binding.get("item", {}).get("value", "")
        rows.append({
            "qid": clean_qid(item),
            "label": binding.get("itemLabel", {}).get("value", ""),
            "wikidata_description": binding.get("itemDescription", {}).get("value", ""),
            "wikipedia_url": binding.get("article", {}).get("value", ""),
            "sitelinks": safe_int(binding.get("sitelinks", {}).get("value", 0)),
        })
    wikidata_cache[query_name] = rows
    save_cache(WIKIDATA_CACHE_JSON, wikidata_cache)
    time.sleep(sleep_seconds)
    return rows

def resolve_title_to_qid(title, subject):
    cache_key = f"title::{title}"
    if cache_key in wikidata_cache:
        payload = wikidata_cache[cache_key]
    else:
        params = urllib.parse.urlencode({
            "action": "query",
            "format": "json",
            "redirects": "1",
            "titles": title,
            "prop": "pageprops|info",
            "inprop": "url",
        })
        url = "https://en.wikipedia.org/w/api.php?" + params
        payload = http_json(url, timeout=60)
        wikidata_cache[cache_key] = payload
        save_cache(WIKIDATA_CACHE_JSON, wikidata_cache)
        time.sleep(0.1)

    pages = payload.get("query", {}).get("pages", {})
    for page in pages.values():
        qid = page.get("pageprops", {}).get("wikibase_item", "")
        if qid:
            return {
                "qid": clean_qid(qid),
                "label": page.get("title", title),
                "wikidata_description": "",
                "wikipedia_url": page.get("fullurl", f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}"),
                "sitelinks": 999999,
                "subject": subject,
                "source_query": "manual_seed_title",
                "source_channel": "manual_seed",
            }
    return None


In [ ]:
def make_subject_query(subject, roots, limit):
    values = " ".join(f"wd:{qid}" for qid in roots)
    return f"""
SELECT DISTINCT ?item ?itemLabel ?itemDescription ?article ?sitelinks WHERE {{
  VALUES ?root {{ {values} }}
  {{ ?item wdt:P31/wdt:P279* ?root . }}
  UNION
  {{ ?item wdt:P279* ?root . }}
  UNION
  {{ ?item wdt:P361 ?root . }}
  UNION
  {{ ?item wdt:P1269 ?root . }}

  ?article schema:about ?item ; schema:isPartOf <https://en.wikipedia.org/> .
  ?item wikibase:sitelinks ?sitelinks .

  FILTER(?sitelinks >= 5)
  FILTER(!STRSTARTS(STR(?article), "https://en.wikipedia.org/wiki/List_of"))
  FILTER(!STRSTARTS(STR(?article), "https://en.wikipedia.org/wiki/Index_of"))
  FILTER(!STRSTARTS(STR(?article), "https://en.wikipedia.org/wiki/Outline_of"))
  FILTER NOT EXISTS {{ ?item wdt:P31 wd:Q5 . }}              # humans / biographies
  FILTER NOT EXISTS {{ ?item wdt:P31 wd:Q4167410 . }}       # disambiguation pages
  FILTER NOT EXISTS {{ ?item wdt:P31 wd:Q13406463 . }}      # Wikimedia list articles
  FILTER NOT EXISTS {{ ?item wdt:P31/wdt:P279* wd:Q16521 . }} # taxa/species; manual seeds add key taxa

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY DESC(?sitelinks)
LIMIT {limit}
"""

EXCLUDE_LABEL_PATTERNS = re.compile(r"\b(list of|index of|outline of|timeline of|history of|glossary of|bibliography of)\b", re.I)

def candidate_is_allowed(row):
    label = normalize_text(row.get("label", ""))
    url = normalize_text(row.get("wikipedia_url", ""))
    if not row.get("qid") or "wikipedia.org/wiki/" not in url:
        return False
    if EXCLUDE_LABEL_PATTERNS.search(label):
        return False
    return True


## 3. Run Wikidata Discovery and Select QIDs

This step is intentionally inspectable. Review `science_nature_wiki_v1_selected_qids.csv` before turning on the DragonLLM fetch.

In [ ]:
import json

cache_path = LOGS_DIR / "science_nature_wiki_v1_wikidata_cache.json"
cache = json.loads(cache_path.read_text())

print("cache entries:", len(cache))

for k in sorted(cache.keys()):
    if k.startswith("subject::"):
        print(k, len(cache[k]))

In [ ]:
wikidata_cache = load_cache(WIKIDATA_CACHE_JSON)

for k in sorted(wikidata_cache.keys()):
    if k.startswith("subject::"):
        print(k, len(wikidata_cache[k]))

In [ ]:
if not RUN_WIKIDATA_DISCOVERY and SELECTED_QIDS_CSV.exists():
    selected = pd.read_csv(SELECTED_QIDS_CSV).fillna("")
    print("loaded selected QIDs:", len(selected))

else:
    candidate_rows = []

    print("Resolving manual seed titles...")
    for subject, titles in MANUAL_SEED_TITLES.items():
        for title in tqdm(titles, desc=f"manual {subject}"):
            resolved = resolve_title_to_qid(title, subject)
            if resolved:
                candidate_rows.append(resolved)

    wikidata_cache = load_cache(WIKIDATA_CACHE_JSON)

    MISSING_BROAD_SUBJECTS = {
      #  "earth_science", troppo pesante, aggiungeremo
        "astronomy",
        "scientific_method",
        "general_science",
    }

    QUERY_LIMIT_OVERRIDES = {
        "earth_science": 300,
        "astronomy": 300,
        "scientific_method": 160,
        "general_science": 200,
    }

    def pick_cached_subject_rows(subject):
        prefix = f"subject::{subject}::limit::"
        keys = [k for k in wikidata_cache.keys() if k.startswith(prefix)]

        if not keys:
            return None, []

        def score_key(k):
            try:
                limit = int(k.rsplit("::", 1)[-1])
            except Exception:
                limit = 0
            return (len(wikidata_cache.get(k, [])), limit)

        best_key = max(keys, key=score_key)
        return best_key, wikidata_cache.get(best_key, [])

    print("Running broad Wikidata discovery...")

    for subject, quota in SUBJECT_QUOTAS.items():
        cached_key, cached_rows = pick_cached_subject_rows(subject)

        if subject not in MISSING_BROAD_SUBJECTS:
            if cached_rows:
                rows = cached_rows
                source_query = cached_key
                print(f"{subject} candidates: {len(rows)} (cache hit: {cached_key})")
            else:
                rows = []
                source_query = ""
                print(f"{subject}: no cache found, skipping broad query")

        else:
            if cached_rows:
                rows = cached_rows
                source_query = cached_key
                print(f"{subject} candidates: {len(rows)} (cache hit: {cached_key})")
            else:
                limit = QUERY_LIMIT_OVERRIDES.get(subject, max(quota * 2, 150))
                query_name = f"subject::{subject}::limit::{limit}"
                query = make_subject_query(subject, SUBJECT_ROOT_QIDS[subject], limit)

            try:
                rows = run_sparql(query_name, query, sleep_seconds=120)
                source_query = query_name
                print(f"{subject} candidates: {len(rows)}")
            except Exception as exc:
                rows = []
                source_query = ""
                print(f"{subject}: broad query failed, using manual seeds only. Error: {exc}")

        for row in rows:
            row = dict(row)
            row.update({
                "subject": subject,
                "source_query": source_query,
                "source_channel": "wikidata_subject_root",
            })
            candidate_rows.append(row)

    candidates = pd.DataFrame(candidate_rows).fillna("")
    candidates["qid"] = candidates["qid"].map(clean_qid)
    candidates["sitelinks"] = candidates["sitelinks"].map(lambda x: safe_int(x, 0))
    candidates = candidates[candidates.apply(candidate_is_allowed, axis=1)].copy()

    candidates["is_manual_seed"] = candidates["source_channel"].eq("manual_seed")
    candidates["selection_score"] = (
        candidates["sitelinks"]
        + candidates["is_manual_seed"].astype(int) * 1_000_000
    )

    candidates = candidates.sort_values(["selection_score", "sitelinks"], ascending=False)
    candidates = candidates.drop_duplicates("qid", keep="first").reset_index(drop=True)

    selected_parts = []
    for subject, quota in SUBJECT_QUOTAS.items():
        subject_df = (
            candidates[candidates["subject"] == subject]
            .sort_values("selection_score", ascending=False)
            .head(quota)
        )
        selected_parts.append(subject_df)

    selected = pd.concat(selected_parts, ignore_index=True).drop_duplicates("qid", keep="first")

    already = set(selected["qid"])
    backfill_rows = []

    for subject, quota in SUBJECT_QUOTAS.items():
        have = int((selected["subject"] == subject).sum())
        need = max(0, quota - have)

        if need:
            pool = candidates[
                (candidates["subject"] == subject)
                & (~candidates["qid"].isin(already))
            ]
            add = pool.sort_values("selection_score", ascending=False).head(need)
            backfill_rows.append(add)
            already.update(add["qid"].tolist())

    if backfill_rows:
        selected = (
            pd.concat([selected] + backfill_rows, ignore_index=True)
            .drop_duplicates("qid", keep="first")
        )

    selected["topic"] = selected["label"].where(
        selected["label"].astype(str).str.len() > 0,
        selected["qid"],
    )
    selected["source_dataset"] = SOURCE_WIKI_DATASET
    selected["source_type"] = WIKI_SOURCE_TYPE
    selected["license"] = WIKIPEDIA_LICENSE
    selected["taxonomy_source"] = TAXONOMY_SOURCE
    selected["taxonomy_confidence"] = selected["is_manual_seed"].map(
        lambda x: 1.0 if bool(x) else 0.85
    )

    candidates.to_csv(CANDIDATE_QIDS_CSV, index=False)
    selected.to_csv(SELECTED_QIDS_CSV, index=False)

print("selected total:", len(selected))
display(selected["subject"].value_counts().rename_axis("subject").reset_index(name="rows"))
display(
    selected[
        ["qid", "label", "subject", "wikipedia_url", "sitelinks", "source_channel", "selection_score"]
    ].head(50)
)
print("saved selected:", SELECTED_QIDS_CSV)

In [ ]:
cache = load_cache(WIKIDATA_CACHE_JSON)

expected = [
    "physics",
    "chemistry",
    "biology",
    "ecology",
    "earth_science",
    "astronomy",
    "scientific_method",
    "general_science",
]

present = set()
for k in cache:
    if k.startswith("subject::"):
        subject = k.split("::")[1]
        present.add(subject)
        print(k, len(cache[k]))

print("missing:", sorted(set(expected) - present))

In [ ]:
wikidata_cache = load_cache(WIKIDATA_CACHE_JSON)

candidate_rows = []

def manual_seed_from_cache(title, subject):
    payload = wikidata_cache.get(f"title::{title}")
    if not payload:
        print("missing manual seed cache:", subject, "|", title)
        return None

    pages = payload.get("query", {}).get("pages", {})
    for page in pages.values():
        qid = page.get("pageprops", {}).get("wikibase_item", "")
        if qid:
            return {
                "qid": clean_qid(qid),
                "label": page.get("title", title),
                "wikidata_description": "",
                "wikipedia_url": page.get(
                    "fullurl",
                    f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}"
                ),
                "sitelinks": 999999,
                "subject": subject,
                "source_query": "manual_seed_title",
                "source_channel": "manual_seed",
            }
    return None

print("Loading manual seed titles from cache...")
for subject, titles in MANUAL_SEED_TITLES.items():
    for title in titles:
        resolved = manual_seed_from_cache(title, subject)
        if resolved:
            candidate_rows.append(resolved)

def pick_cached_subject_rows(subject):
    prefix = f"subject::{subject}::limit::"
    keys = [k for k in wikidata_cache.keys() if k.startswith(prefix)]

    if not keys:
        return None, []

    def score_key(k):
        try:
            limit = int(k.rsplit("::", 1)[-1])
        except Exception:
            limit = 0
        return (len(wikidata_cache.get(k, [])), limit)

    best_key = max(keys, key=score_key)
    return best_key, wikidata_cache.get(best_key, [])

print("Loading broad Wikidata candidates from cache only...")

for subject, quota in SUBJECT_QUOTAS.items():
    cached_key, rows = pick_cached_subject_rows(subject)

    if rows:
        print(f"{subject} candidates: {len(rows)} (cache hit: {cached_key})")
    else:
        print(f"{subject}: no broad cache, using manual seeds only")
        rows = []

    for row in rows:
        row = dict(row)
        row.update({
            "subject": subject,
            "source_query": cached_key or "",
            "source_channel": "wikidata_subject_root",
        })
        candidate_rows.append(row)

candidates = pd.DataFrame(candidate_rows).fillna("")
candidates["qid"] = candidates["qid"].map(clean_qid)
candidates["sitelinks"] = candidates["sitelinks"].map(lambda x: safe_int(x, 0))
candidates = candidates[candidates.apply(candidate_is_allowed, axis=1)].copy()

candidates["is_manual_seed"] = candidates["source_channel"].eq("manual_seed")
candidates["selection_score"] = (
    candidates["sitelinks"]
    + candidates["is_manual_seed"].astype(int) * 1_000_000
)

candidates = candidates.sort_values(["selection_score", "sitelinks"], ascending=False)
candidates = candidates.drop_duplicates("qid", keep="first").reset_index(drop=True)

selected_parts = []
for subject, quota in SUBJECT_QUOTAS.items():
    subject_df = (
        candidates[candidates["subject"] == subject]
        .sort_values("selection_score", ascending=False)
        .head(quota)
    )
    selected_parts.append(subject_df)

selected = pd.concat(selected_parts, ignore_index=True).drop_duplicates("qid", keep="first")

selected["topic"] = selected["label"].where(
    selected["label"].astype(str).str.len() > 0,
    selected["qid"],
)
selected["source_dataset"] = SOURCE_WIKI_DATASET
selected["source_type"] = WIKI_SOURCE_TYPE
selected["license"] = WIKIPEDIA_LICENSE
selected["taxonomy_source"] = TAXONOMY_SOURCE
selected["taxonomy_confidence"] = selected["is_manual_seed"].map(
    lambda x: 1.0 if bool(x) else 0.85
)

candidates.to_csv(CANDIDATE_QIDS_CSV, index=False)
selected.to_csv(SELECTED_QIDS_CSV, index=False)

print("candidate total:", len(candidates))
print("selected total:", len(selected))
display(selected["subject"].value_counts().rename_axis("subject").reset_index(name="rows"))
display(
    selected[
        ["qid", "label", "subject", "wikipedia_url", "sitelinks", "source_channel", "selection_score"]
    ].head(80)
)
print("saved candidates:", CANDIDATE_QIDS_CSV)
print("saved selected:", SELECTED_QIDS_CSV)

## 4. Fetch Articles from DragonLLM and Save Dataset

This is the long dataset-build step. It streams `DragonLLM/Clean-Wikipedia-English-Articles` and keeps rows whose `entity`/QID is in the selected target set.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
def row_value(row, *keys, default=""):
    for key in keys:
        if key in row and row[key] not in [None, ""]:
            return row[key]
    return default

def jsonish(value):
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False)
    return normalize_text(value)

def make_article_row(source_row, meta):
    qid = clean_qid(row_value(source_row, "qid", "entity", default=meta.get("qid", ""))) or meta.get("qid", "")
    title = normalize_text(row_value(source_row, "title", default=meta.get("label", "")))
    url = canonical_url(row_value(source_row, "url", "canonicalurl", default=meta.get("wikipedia_url", ""))) or canonical_url(meta.get("wikipedia_url", ""))
    text = normalize_text(row_value(source_row, "text", default=""))
    doc_id = f"wiki_{qid}" if qid else f"wiki_{normalize_title(title).replace(' ', '_')}"
    return {
        "doc_id": doc_id,
        "text": text,
        "title": title,
        "url": url,
        "qid": qid,
        "source_dataset": SOURCE_WIKI_DATASET,
        "source_type": WIKI_SOURCE_TYPE,
        "license": WIKIPEDIA_LICENSE,
        "subject": meta.get("subject", "general_science"),
        "topic": meta.get("topic", meta.get("label", title)),
        "taxonomy_source": meta.get("taxonomy_source", TAXONOMY_SOURCE),
        "taxonomy_confidence": float(meta.get("taxonomy_confidence", 0.85) or 0.85),
        "wikidata_label": meta.get("label", title),
        "wikidata_description": meta.get("wikidata_description", ""),
        "wikipedia_url": meta.get("wikipedia_url", url),
        "sitelinks": safe_int(meta.get("sitelinks", 0)),
        "source_query": meta.get("source_query", ""),
        "source_channel": meta.get("source_channel", ""),
        "categories": jsonish(row_value(source_row, "categories", default="")),
        "infobox": jsonish(row_value(source_row, "infobox", default="")),
        "token_count": safe_int(row_value(source_row, "token_count", default=0)),
        "revdate": normalize_text(row_value(source_row, "revdate", default="")),
    }

if not RUN_FETCH_AND_BUILD_DATASET:
    print("Skipping dataset fetch/build because RUN_FETCH_AND_BUILD_DATASET=False")
else:
    if OUTPUT_DATASET_DIR.exists():
        if not OVERWRITE_OUTPUT_DATASET:
            raise FileExistsError(f"{OUTPUT_DATASET_DIR} exists. Set OVERWRITE_OUTPUT_DATASET=True to replace it.")
        shutil.rmtree(OUTPUT_DATASET_DIR)

    selected = pd.read_csv(SELECTED_QIDS_CSV).fillna("")
    selected["qid"] = selected["qid"].map(clean_qid)
    target_meta = {row["qid"]: row.to_dict() for _, row in selected.iterrows() if row.get("qid")}
    target_qids = set(target_meta)
    found_by_qid = {}

    print("target QIDs:", len(target_qids))
    source_stream = load_dataset(SOURCE_WIKI_DATASET, split="train", streaming=True)

    for idx, row in enumerate(source_stream, start=1):
        qid = clean_qid(row_value(row, "qid", "entity", default=""))
        if qid in target_qids and qid not in found_by_qid:
            found_by_qid[qid] = row
            meta = target_meta[qid]
            print(f"found {len(found_by_qid):>4}/{len(target_qids)} | {qid} | {meta.get('label', row.get('title', ''))}")
            if len(found_by_qid) == len(target_qids):
                break

        if idx % 100000 == 0:
            print(f"scanned {idx:,} rows | found {len(found_by_qid):,}/{len(target_qids):,}")
        if MAX_SOURCE_ROWS_TO_SCAN and idx >= MAX_SOURCE_ROWS_TO_SCAN:
            print("stopping at MAX_SOURCE_ROWS_TO_SCAN:", MAX_SOURCE_ROWS_TO_SCAN)
            break

    article_rows = []
    fetch_rows = []
    for qid, meta in target_meta.items():
        if qid in found_by_qid:
            article = make_article_row(found_by_qid[qid], meta)
            if article["text"]:
                article_rows.append(article)
                status = "added"
            else:
                status = "empty_text_skipped"
        else:
            status = "not_found_in_dragonllm"
        fetch_rows.append({
            "qid": qid,
            "label": meta.get("label", ""),
            "subject": meta.get("subject", ""),
            "wikipedia_url": meta.get("wikipedia_url", ""),
            "status": status,
        })

    dataset = Dataset.from_list(article_rows)
    DatasetDict({"train": dataset}).save_to_disk(str(OUTPUT_DATASET_DIR))
    pd.DataFrame(fetch_rows).to_csv(FETCH_REPORT_CSV, index=False)

    build_report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_dataset": SOURCE_WIKI_DATASET,
        "output_dataset_dir": str(OUTPUT_DATASET_DIR),
        "selected_qids_csv": str(SELECTED_QIDS_CSV),
        "fetch_report_csv": str(FETCH_REPORT_CSV),
        "target_qids": len(target_qids),
        "found_qids": len(found_by_qid),
        "final_rows": len(article_rows),
        "subject_counts": pd.DataFrame(article_rows)["subject"].value_counts().to_dict() if article_rows else {},
        "fields": list(article_rows[0].keys()) if article_rows else [],
    }
    BUILD_REPORT_JSON.write_text(json.dumps(build_report, indent=2, ensure_ascii=False), encoding="utf-8")

    print("saved dataset:", OUTPUT_DATASET_DIR)
    print("saved fetch report:", FETCH_REPORT_CSV)
    print(json.dumps(build_report, indent=2, ensure_ascii=False))


## 5. Dataset Audit

In [ ]:
if OUTPUT_DATASET_DIR.exists():
    ds = load_from_disk(str(OUTPUT_DATASET_DIR))["train"]
    wiki_df = ds.to_pandas().fillna("")
    print("dataset rows:", len(wiki_df))
    display(wiki_df["subject"].value_counts().rename_axis("subject").reset_index(name="rows"))
    display(wiki_df[["qid", "title", "subject", "topic", "token_count", "url"]].head(50))
else:
    print("Dataset does not exist yet:", OUTPUT_DATASET_DIR)


## 6. Optional Ollama Preflight

Index build requires the embedding endpoint. If Ollama is already running, leave both flags as `False` and just run the preflight.

In [ ]:
!ollama --version

In [ ]:
!apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
import subprocess, time

subprocess.run(["killall", "ollama"], check=False)

with open("/tmp/ollama_science_wiki_index.log", "ab") as log_file:
    subprocess.Popen(
        ["bash", "-lc", "OLLAMA_CONTEXT_LENGTH=8192 ollama serve"],
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(5)

In [ ]:
EMBEDDING_MODEL = "hf.co/unsloth/embeddinggemma-300m-GGUF:BF16"

!ollama pull {EMBEDDING_MODEL}
!ollama list

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")
resp = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="photosynthesis is a biological process",
)

print(len(resp.data[0].embedding))

In [ ]:
EMBEDDING_MODEL = "hf.co/unsloth/embeddinggemma-300m-GGUF:BF16"
EMBEDDING_BASE_URL = "http://localhost:11434/v1/"
EMBEDDING_API_KEY = "ollama"

START_OLLAMA_SERVER = False
PULL_EMBEDDING_MODEL = False

if START_OLLAMA_SERVER:
    import subprocess
    subprocess.run(["killall", "ollama"], check=False)
    with open("/tmp/ollama_science_wiki_index.log", "ab") as log_file:
        subprocess.Popen(["bash", "-lc", "OLLAMA_CONTEXT_LENGTH=8192 ollama serve"], stdout=log_file, stderr=log_file)
    time.sleep(5)

if PULL_EMBEDDING_MODEL:
    import subprocess
    subprocess.run(["ollama", "pull", EMBEDDING_MODEL], check=True)

def assert_embedding_endpoint_ready():
    tags_url = EMBEDDING_BASE_URL.rstrip("/")
    if tags_url.endswith("/v1"):
        tags_url = tags_url[:-3]
    tags_url = tags_url.rstrip("/") + "/api/tags"
    try:
        payload = http_json(tags_url, timeout=5)
    except Exception as exc:
        raise RuntimeError(
            "Embedding endpoint is not reachable. Start Ollama and pull the embedding model before indexing.\n"
            f"Tried: {tags_url}\n"
            f"Model: {EMBEDDING_MODEL}"
        ) from exc
    print("embedding endpoint reachable:", tags_url)
    print(json.dumps(payload, indent=2, ensure_ascii=False)[:1000])

if RUN_INDEX_BUILD:
    assert_embedding_endpoint_ready()
else:
    print("Skipping endpoint preflight because RUN_INDEX_BUILD=False")


## 7. Build FAISS Index

Wikipedia articles are chunked at 512 tokens with 128 overlap.

In [ ]:
!pip install langchain_core langchain_openai langchain_community chromadb

In [ ]:
!pip install -q faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
if not RUN_INDEX_BUILD:
    print("Skipping index build because RUN_INDEX_BUILD=False")
else:
    from langchain_core.documents import Document
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_text_splitters import TokenTextSplitter

    if not OUTPUT_DATASET_DIR.exists():
        raise FileNotFoundError(f"Build dataset first: {OUTPUT_DATASET_DIR}")
    if OUTPUT_INDEX_DIR.exists():
        if not OVERWRITE_INDEX:
            raise FileExistsError(f"{OUTPUT_INDEX_DIR} exists. Set OVERWRITE_INDEX=True to replace it.")
        shutil.rmtree(OUTPUT_INDEX_DIR)

    assert_embedding_endpoint_ready()

    ds = load_from_disk(str(OUTPUT_DATASET_DIR))["train"]
    df = ds.to_pandas().fillna("")

    splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=512, chunk_overlap=128)
    metadata_cols = [
        "doc_id", "title", "url", "qid", "source_dataset", "source_type", "license",
        "subject", "topic", "taxonomy_source", "taxonomy_confidence", "wikidata_label",
        "wikidata_description", "wikipedia_url", "sitelinks", "token_count", "revdate",
    ]

    docs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="chunking wiki articles"):
        text = str(row.get("text", "") or "")
        if not text.strip():
            continue
        base_meta = {col: row.get(col, "") for col in metadata_cols if col in df.columns}
        try:
            base_meta["taxonomy_confidence"] = float(base_meta.get("taxonomy_confidence", 0.0) or 0.0)
        except Exception:
            base_meta["taxonomy_confidence"] = 0.0
        chunks = splitter.split_text(text)
        for chunk_idx, chunk in enumerate(chunks):
            meta = dict(base_meta)
            meta["chunk_id"] = f"{base_meta.get('doc_id', '')}_chunk_{chunk_idx:04d}"
            meta["chunk_index"] = chunk_idx
            docs.append(Document(page_content=chunk, metadata=meta))

    print("chunks to embed:", len(docs))
    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=EMBEDDING_BASE_URL,
        api_key=EMBEDDING_API_KEY,
        check_embedding_ctx_length=False,
    )

    vectorstore = None
    batch_size = 64
    for start in tqdm(range(0, len(docs), batch_size), desc="embedding wiki chunks"):
        batch = docs[start:start + batch_size]
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)

    vectorstore.save_local(str(OUTPUT_INDEX_DIR))
    index_report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "input_dataset_dir": str(OUTPUT_DATASET_DIR),
        "output_index_dir": str(OUTPUT_INDEX_DIR),
        "article_rows": len(df),
        "chunks": len(docs),
        "chunk_size": 512,
        "chunk_overlap": 128,
        "embedding_model": EMBEDDING_MODEL,
        "metadata_cols": metadata_cols + ["chunk_id", "chunk_index"],
        "subject_counts": df["subject"].value_counts().to_dict(),
        "index_files": sorted(p.name for p in OUTPUT_INDEX_DIR.iterdir()),
    }
    INDEX_REPORT_JSON.write_text(json.dumps(index_report, indent=2, ensure_ascii=False), encoding="utf-8")
    print("saved index:", OUTPUT_INDEX_DIR)
    print(json.dumps(index_report, indent=2, ensure_ascii=False))


## 8. Smoke Test

In [ ]:
if OUTPUT_INDEX_DIR.exists():
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, base_url=EMBEDDING_BASE_URL, api_key=EMBEDDING_API_KEY, check_embedding_ctx_length=False)
    store = FAISS.load_local(str(OUTPUT_INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
    queries = [
        "What is string theory?",
        "What is ionic bonding between sodium and chlorine?",
        "What is photosynthesis?",
        "What is an ecosystem food chain?",
    ]
    rows = []
    for query in queries:
        for rank, (doc, score) in enumerate(store.similarity_search_with_score(query, k=5), start=1):
            rows.append({
                "query": query,
                "rank": rank,
                "score": float(score),
                "title": doc.metadata.get("title", ""),
                "subject": doc.metadata.get("subject", ""),
                "topic": doc.metadata.get("topic", ""),
                "qid": doc.metadata.get("qid", ""),
                "text_preview": doc.page_content[:300],
            })
    display(pd.DataFrame(rows))
else:
    print("Index does not exist yet:", OUTPUT_INDEX_DIR)
